# AL-2002: Artificial Intelligence — Lab Manual 06
## FAST Timetable Scheduler — Genetic Algorithm
**National University of Computer and Emerging Sciences (FAST-NU), Lahore**

| | |
|---|---|
| **Name** | Zohaib Hussain |
| **Roll Number** | 23L-3087 |

---
## Objectives
- Implement a **Genetic Algorithm** to solve a real-world scheduling problem
- Design chromosome encoding for discrete problems
- Implement and test multiple constraint types in a fitness function
- Observe how search space complexity affects GA convergence

---
# Part A — FAST Timetable Scheduler
## Problem Statement
FAST-NU needs to schedule **10 courses** for the upcoming semester. You are given:
- 3 teachers
- 3 rooms (one dedicated lab room)
- 4 time slots
- 3 student batches

**Hard Constraints:**
1. No teacher is assigned to two courses in the same slot
2. No room is used for two courses in the same slot

## Setup

In [1]:
import random
from collections import Counter

# Optional: set a seed for reproducible results across runs
random.seed(42)

courses  = ['CS301', 'CS302', 'CS303', 'CS401', 'CS402', 'CS403',
            'SE301', 'SE302', 'DS401', 'DS402']

teachers = ['Dr_Sobia', 'Dr_Aamir', 'Dr_Asif']
rooms    = ['Room_A', 'Room_B', 'Lab_1']
slots    = ['8am', '10am', '12pm', '2pm']
batches  = ['CS-A', 'CS-B', 'SE-A']

# Each course belongs to a batch
course_batch = {
    'CS301': 'CS-A', 'CS302': 'CS-A', 'CS303': 'CS-A',
    'CS401': 'CS-B', 'CS402': 'CS-B', 'CS403': 'CS-B',
    'SE301': 'SE-A', 'SE302': 'SE-A',
    'DS401': 'CS-A', 'DS402': 'CS-B'
}

# Lab courses must be assigned to Lab_1 only (used in Part B)
lab_courses = ['CS303', 'SE302', 'DS401']

print('Setup complete.')
print(f'Courses: {len(courses)} | Teachers: {len(teachers)} | Rooms: {len(rooms)} | Slots: {len(slots)}')

Setup complete.
Courses: 10 | Teachers: 3 | Rooms: 3 | Slots: 4


## Task 1 — `generate_chromosome()`
Generate a random chromosome. Each **gene** is a tuple:
`(course, teacher, room, slot)`

One gene per course, so the chromosome has 10 genes.

In [2]:
def generate_chromosome():
    """
    Generate a random chromosome representing a timetable.
    
    Returns:
        list of tuples: Each tuple is (course, teacher, room, slot)
                        One gene per course (10 genes total).
    """
    # TODO: Implement this function
    # Hint: For each course, randomly pick a teacher, room, and slot
    chromosome = []
    for i in range(10):
        cource, batch = random.choice(list(course_batch.items()))
        teacher = random.choice(teachers)
        slot = random.choice(slots)
        # if course also has a lab
        if cource in lab_courses:
            room = random.choice(rooms)
        else:
            room = random.choice(rooms[:-1]) # as last room is lab
        chromosome.append((cource, teacher, room, slot))
    return chromosome


# Test
sample = generate_chromosome()
print('Sample chromosome:')
for gene in sample:
    print(f'  {gene}')

Sample chromosome:
  ('CS302', 'Dr_Sobia', 'Room_A', '12pm')
  ('CS401', 'Dr_Sobia', 'Room_A', '8am')
  ('DS402', 'Dr_Aamir', 'Room_A', '8am')
  ('CS302', 'Dr_Sobia', 'Room_A', '10am')
  ('DS401', 'Dr_Sobia', 'Room_A', '2pm')
  ('SE302', 'Dr_Asif', 'Room_A', '12pm')
  ('CS303', 'Dr_Asif', 'Room_B', '2pm')
  ('CS402', 'Dr_Sobia', 'Room_B', '10am')
  ('CS302', 'Dr_Sobia', 'Room_A', '2pm')
  ('CS403', 'Dr_Aamir', 'Room_A', '12pm')


## Task 2 — `fitness(chromosome)`
Calculate the fitness of a chromosome. Start at **0** and **penalise** each constraint violation.

**Constraints to check:**
- `-10` for each *extra* course beyond the first that shares the same teacher+slot
- `-10` for each *extra* course beyond the first that shares the same room+slot

> **Example:** If Dr_Sobia is assigned 3 courses all in `8am`, that is 2 violations → `-20`.

A **fitness of 0** means no violations (perfect timetable). More negative = worse.

In [3]:
def fitness(chromosome):
    """
    Evaluate the fitness of a chromosome.
    
    Args:
        chromosome (list): List of (course, teacher, room, slot) tuples.
    
    Returns:
        int: Fitness score. 0 = perfect; more negative = more violations.
    """
    score = 0

    # Hard Constraint 1: No teacher clash (same teacher, same slot)
    # Hint: Build a list of (teacher, slot) pairs from the chromosome.
    #       Use Counter to find duplicates, and subtract 10 for each duplicate.
    # TODO: Detect and penalise teacher clashes
    
    # Hard Constraint 2: No room clash (same room, same slot)
    # Hint: Same approach, but with (room, slot) pairs.
    # TODO: Detect and penalise room clashes

    teacher_slot = []
    room_slot = []

    for chrm in chromosome:
        teacher_slot.append((chrm[1], chrm[3]))
        room_slot.append((chrm[2], chrm[3]))

    count_ts, count_rs = Counter(teacher_slot), Counter(room_slot)
    duplicates_ts, duplicates_rs = count_ts.total() - len(count_ts), count_rs.total() - len(count_rs)

    score -= duplicates_ts * 10 + duplicates_rs * 10

    return score


# Test
sample = generate_chromosome()
print(f'Sample fitness: {fitness(sample)}')

Sample fitness: -60


## Task 3 — `run_ga()`
Implement the main Genetic Algorithm loop.

**Steps:**
1. Initialise a population of random chromosomes
2. Evaluate fitness of each
3. Select parents (e.g. tournament selection)
4. Crossover to produce offspring
5. Mutate offspring randomly
6. Repeat until a chromosome with fitness = 0 is found (or max generations reached)

**Helper functions** are provided below for selection, crossover, and mutation. Implement each one, then wire them together in `run_ga()`.

> **Important:** `run_ga()` must return a **tuple** `(best_chromosome, generation_found)` — the best chromosome found and the generation it was found at (or `-1` if no perfect solution was reached). Part B unpacks this directly, e.g. `best, gen = run_ga(...)`.

In [4]:
def tournament_select(population, fitness_func, k=5):
    """
    Select one parent using tournament selection.
    
    Pick k random chromosomes from the population and return the one
    with the highest (least negative) fitness.
    
    Args:
        population (list): List of chromosomes.
        fitness_func (callable): Fitness function.
        k (int): Tournament size.
    
    Returns:
        list: The winning chromosome.
    """
    # TODO: Implement tournament selection
    # Hint: Use random.sample(population, k) to pick candidates,
    #       then return the one with the best fitness_func score.
    
    best_chrms = random.sample(population, k)
    max_fitness = float("-inf")
    max_chrm = None

    for chrm in best_chrms:
        fittness = fitness_func(chrm)
        if fittness > max_fitness:
            max_fitness = fittness
            max_chrm = chrm

    return max_chrm

def crossover(parent1, parent2):
    """
    Produce two offspring via single-point crossover.
    
    Pick a random crossover point, then swap the gene segments.
    
    Args:
        parent1 (list): First parent chromosome.
        parent2 (list): Second parent chromosome.
    
    Returns:
        tuple: (child1, child2)
    """
    # TODO: Implement single-point crossover
    point = random.randint(1, len(parent1) - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent1[point:] + parent2[:point]

    return (child1, child2)

def mutate(chromosome, mutation_rate=0.1):
    """
    Randomly mutate genes in a chromosome.
    
    For each gene, with probability mutation_rate, replace the
    teacher, room, and slot with random new values.
    
    Args:
        chromosome (list): Chromosome to mutate (modified in place).
        mutation_rate (float): Probability of mutating each gene.
    
    Returns:
        list: The (possibly mutated) chromosome.
    """
    # TODO: Implement mutation
    # Hint: For each gene index, if random.random() < mutation_rate,
    #       replace that gene with a new random (course, teacher, room, slot).
    #       Keep the course the same; only randomise teacher, room, slot.
    for gene in chromosome:
        if random.random() < mutation_rate:
            course = gene[0]
            teacher = random.choice(teachers)
            slot = random.choice(slots)
            # if course also has a lab
            if course in lab_courses:
                room = random.choice(rooms)
            else:
                room = random.choice(rooms[:-1]) # as last room is lab
            gene = (course, teacher, room, slot)
    return chromosome

In [5]:
def run_ga(population_size=500, max_generations=2000, mutation_rate=0.1, fitness_func=fitness):
    """
    Run the Genetic Algorithm to find a valid timetable.
    
    Args:
        population_size (int): Number of chromosomes in each generation.
        max_generations (int): Maximum number of generations to run.
        mutation_rate (float): Probability of mutating a gene.
        fitness_func (callable): Fitness function to evaluate chromosomes.
    
    Returns:
        tuple: (best_chromosome, generation_found)
               generation_found = -1 if no valid solution found.
    """
    # Step 1: Initialise population
    population = [generate_chromosome() for _ in range(population_size)]

    for generation in range(max_generations):
        # Step 2: Evaluate fitness
        # TODO: Score each chromosome using fitness_func

        # Step 3: Check for perfect solution (fitness == 0)
        # TODO: Return (best_chromosome, generation) if found

        for chromosome in population:
            if fitness_func(chromosome) == 0:
                return chromosome, generation

        # Step 4: Selection + Crossover + Mutation
        # TODO: Build a new population using tournament_select, crossover, mutate
        # 	
        # Suggested approach:
        new_population = []
        while len(new_population) < population_size:
            p1 = tournament_select(population, fitness_func)
            p2 = tournament_select(population, fitness_func)
            c1, c2 = crossover(p1, p2)
            new_population.append(mutate(c1, mutation_rate))
            new_population.append(mutate(c2, mutation_rate))
            
        population = new_population[:population_size]

    # Return best found even if not perfect
    best = max(population, key=fitness_func)
    return best, -1


best_chromosome, gen_found = run_ga()
print(f'Solution found at generation: {gen_found}')
print(f'Fitness of best solution: {fitness(best_chromosome)}')

Solution found at generation: 4
Fitness of best solution: 0


## Task 4 — Print the Timetable
Display the resulting timetable in a readable format.

In [6]:
def print_timetable(chromosome):
    """
    Print the timetable in a formatted table.
    
    Args:
        chromosome (list): List of (course, teacher, room, slot) tuples.
    """
    print(f"{'Course':<10} {'Teacher':<10} {'Room':<10} {'Slot':<8}")
    print('-' * 44)
    for gene in chromosome:
        for att in gene:
            print(f"{att:<10}",end='')

        print("")
    print('-' *48)


print_timetable(best_chromosome)

Course     Teacher    Room       Slot    
--------------------------------------------
SE301     Dr_Aamir  Room_B    8am       
SE302     Dr_Aamir  Lab_1     10am      
CS303     Dr_Asif   Room_A    2pm       
CS401     Dr_Sobia  Room_A    10am      
SE302     Dr_Aamir  Lab_1     2pm       
CS403     Dr_Sobia  Room_B    2pm       
CS402     Dr_Sobia  Room_A    12pm      
DS401     Dr_Asif   Room_A    8am       
CS303     Dr_Asif   Room_B    10am      
SE302     Dr_Sobia  Lab_1     8am       
------------------------------------------------


## Results — Part A

| Metric | Your Result |
|--------|-------------|
| Generation at which valid timetable was found |4|
| Any remaining clashes? |No|

---
# Part B — Scaling Up
## 2.1 New Setup
Extend the problem with a 4th teacher and additional rooms.

In [7]:
# Extended setup for Part B
teachers = ['Dr_Sobia', 'Dr_Aamir', 'Dr_Asif', 'Dr_Ali']
rooms    = ['Room_A', 'Room_B', 'Room_C', 'Room_D', 'Lab_1']

print(f'Extended — Teachers: {teachers}')
print(f'Extended — Rooms: {rooms}')

Extended — Teachers: ['Dr_Sobia', 'Dr_Aamir', 'Dr_Asif', 'Dr_Ali']
Extended — Rooms: ['Room_A', 'Room_B', 'Room_C', 'Room_D', 'Lab_1']


## 2.2 Constraint 1 — Batch Clash
A student batch **cannot** have two courses in the same slot.
Penalise `-10` per batch clash.

In [8]:
def fitness_b1(chromosome):
    """_________________ 	_
    Fitness function with Part A constraints + Batch Clash.
    
    Args:
        chromosome (list): List of (course, teacher, room, slot) tuples.
    
    Returns:
        int: Fitness score (0 = perfect).
    """
    # Start with Part A constraints (teacher clash + room clash)
    score = fitness(chromosome)

    # Constraint 1 — Batch clash (same batch, same slot)
    batch_slots = [(course_batch[gene[0]], gene[3]) for gene in chromosome]
    # penalise -10 for each extra course beyond the first in any (batch, slot) pair

    count_bs = Counter(batch_slots)
    duplicates_bs = count_bs.total() - len(count_bs)

    score -= duplicates_bs * 10
    return score


best_b1, gen_b1 = run_ga(fitness_func=fitness_b1)
print(f'Constraint 1 — Generation found: {gen_b1}')
print(f'Fitness: {fitness_b1(best_b1)}')

Constraint 1 — Generation found: 5
Fitness: 0


### Results — Constraint 1

| Metric | Result |
|--------|--------|
| Generations to valid solution |Yes|
| Faster or slower than Part A? |slower|

## 2.3 Constraint 2 — Room Capacity
Lab courses must only be assigned to `Lab_1`.
Penalise `-10` per lab course placed in a regular room.

In [18]:
def fitness_b2(chromosome):
    """
    Fitness with Part A + Batch Clash + Room Capacity constraints.
    
    Args:
        chromosome (list): List of (course, teacher, room, slot) tuples.
    
    Returns:
        int: Fitness score (0 = perfect).
    """
    # Start with all previous constraints
    score = fitness_b1(chromosome)

    # Constraint 2 — Room capacity (lab courses must use Lab_1)
    for gene in chromosome:
        if gene[0] in lab_courses:
            # TODO: Penalise if gene[2] (room) is not 'Lab_1'
            if gene[2] != rooms[-1]:
                score -= 10

    return score


best_b2, gen_b2 = run_ga(fitness_func=fitness_b2)
print(f'Constraint 2 — Generation found: {gen_b2}')
print(f'Fitness: {fitness_b2(best_b2)}')
print(f'Lab courses correctly placed: {sum(1 for g in best_b2 if g[0] in lab_courses and g[2] == "Lab_1")}/{len(lab_courses)}')

Constraint 2 — Generation found: 5
Fitness: 0
Lab courses correctly placed: 2/3


### Results — Constraint 2

| Metric | Result |
|--------|--------|
| Generations to valid solution |yes|
| Faster or slower than Constraint 1? |slower|
| How many lab courses correctly placed? |2|

## 2.4 Constraint 3 — Teacher Load
No teacher should be assigned more than **3 courses** in total.
Penalise `-10` for each course over the limit.

In [25]:
def fitness_b3(chromosome):
    """
    Fitness with all previous constraints + Teacher Load.
    
    Args:
        chromosome (list): List of (course, teacher, room, slot) tuples.
    
    Returns:
        int: Fitness score (0 = perfect).
    """
    # Start with all previous constraints
    score = fitness_b2(chromosome)

    # Constraint 3 — Teacher load (max 3 courses per teacher)
    teacher_counts = Counter([gene[1] for gene in chromosome])
    for teacher, count in teacher_counts.items():
        if count > 3:
            # TODO: Penalise for each course over the limit
            score -= 10 * (count - 3)

    return score


best_b3, gen_b3 = run_ga(fitness_func=fitness_b3)
print(f'Constraint 3 — Generation found: {gen_b3}')
print(f'Fitness: {fitness_b3(best_b3)}')
teacher_counts = Counter([g[1] for g in best_b3])
print('Teacher loads:', dict(teacher_counts))

Constraint 3 — Generation found: 7
Fitness: 0
Teacher loads: {'Dr_Aamir': 2, 'Dr_Sobia': 2, 'Dr_Ali': 3, 'Dr_Asif': 3}


### Results — Constraint 3

| Metric | Result |
|--------|--------|
| Generations to valid solution |7|
| Which teacher was most overloaded before this constraint? |Dr Sobia|
| Is any teacher assigned more than 3 courses after this constraint? |No|

## 2.5 Constraint 4 — Consecutive Slots
No teacher should have more than **2 back-to-back slots**.
Penalise `-10` per teacher exceeding consecutive slot limit.

In [31]:
slot_order = {'8am': 0, '10am': 1, '12pm': 2, '2pm': 3}

def count_consecutive(teacher, chromosome):
    """
    Count how many consecutive slot runs of length > 2 a teacher has.
    
    Args:
        teacher (str): Teacher name to check.
        chromosome (list): Full chromosome.
    
    Returns:
        int: Number of runs longer than 2 consecutive slots.
             Multiply by -10 in fitness_b4 to get the penalty.
    """
    # Steps:
    #   1. Get all slots for this teacher from the chromosome
    curr_tchr_slots = []
    for gene in chromosome:
        tchr, slot = gene[1], gene[3]
        if tchr == teacher:
            curr_tchr_slots.append(slot)
            
    #   2. Convert each slot string to its index using slot_order
    for i, slot in enumerate(curr_tchr_slots):
        curr_tchr_slots[i] = slot_order[slot]
    #   3. Sort the indices
    curr_tchr_slots.sort()
    #   4. Walk through and count runs; if any run exceeds length 2, that's a violation
    #
    # Example: if a teacher has slots [0, 1, 2] that's one run of length 3 → 1 violation
    #          if a teacher has slots [0, 1, 3] that's two runs (0,1) and (3) → 0 violations
    #
    # TODO: Implement this
    if len(curr_tchr_slots) < 2:
        return 0
    count = 0
    for i in range(2, len(curr_tchr_slots)):
        if (curr_tchr_slots[i - 1] == curr_tchr_slots[i] - 1 and
            curr_tchr_slots[i - 2] == curr_tchr_slots[i] - 2):
            count += 1
    return count

def fitness_b4(chromosome):
    """
    Fitness with all previous constraints + Consecutive Slots.
    
    Args:
        chromosome (list): List of (course, teacher, room, slot) tuples.
    
    Returns:
        int: Fitness score (0 = perfect).
    """
    # Start with all previous constraints
    score = fitness_b3(chromosome)

    # Constraint 4 — Consecutive slots (max 2 back-to-back per teacher)
    for teacher in teachers:
        # TODO: Use count_consecutive and penalise -10 per violation
        score -= 10 * count_consecutive(teacher, chromosome)

    return score


results = []
for run in range(3):
    best_b4, gen_b4 = run_ga(fitness_func=fitness_b4)
    results.append((gen_b4, fitness_b4(best_b4)))
    print(f'Run {run+1} — Generation: {gen_b4}, Fitness: {fitness_b4(best_b4)}')

Run 1 — Generation: 11, Fitness: 0
Run 2 — Generation: 12, Fitness: 0
Run 3 — Generation: 10, Fitness: 0


### Results — Constraint 4

| Metric | Result |
|--------|--------|
| Generations to valid solution |yes|
| Faster or slower than Constraint 3? |slower|
| Run GA 3 times — same timetable each time? |no|

---
## 2.6 Final Comparison Table

Fill this in after running all constraints:

| Stage | Constraints Active | Generations to Solution |
|-------|-------------------|------------------------|
| Part A | Teacher clash, Room clash |4|
| Part B — Step 1 | + Batch clash |5|
| Part B — Step 2 | + Room capacity |5|
| Part B — Step 3 | + Teacher load |7|
| Part B — Step 4 | + Consecutive slots |11|